# 00 — Patent Filter: Shrouded vs Open Rotor

Binary visual classification experiment for DINOv2.

| Class | First CPC codes | Count |
|---|---|---|
| `shrouded` | B64C27/20, B64C11/001, B64U50/14 | 75 (all) |
| `open_rotor` | B64C29/0033, B64C29/0025 | 75 (sampled, seed=42) |

**Output:** `selected_patents.csv` → `1627/Shrouded vs Openrotor/`

**Cells:**
1. Load dataset & show statistics
2. Filter by first CPC → assign categories → sample open_rotor
3. Save `selected_patents.csv`
4. Summary: counts per category + sample rows

In [1]:
import sys; sys.path.insert(0, "..")
import pandas as pd
from src.config_loader import load_config
from src.patents import load_patents

cfg = load_config()
df, missing_df = load_patents(cfg)

print(f"Total patents with valid PDF URL : {len(df)}")
print(f"Patents with no PDF URL          : {len(missing_df)}\n")

print("Record Types:")
print(df["Record Type"].value_counts().to_string())

print("\nLegal Status:")
print(df["Family Legal Status(Dead/Alive)"].value_counts().to_string())

# First CPC uses ';' as separator in this dataset
first_cpc = (
    df["CPC"].fillna("")
    .str.split(r"\s*;\s*", n=1).str[0]
    .str.strip()
    .str.replace(r"\s+", "", regex=True)
)
print("\nFirst CPC (codes with ≥ 10 patents):")
cpc_counts = first_cpc.value_counts()
print(cpc_counts[cpc_counts >= 10].to_string())

Loading Excel: /mnt/storage_11tb/Drive_files_to_syncronize/2 - Patente & Validation/3 -Raw_Patent_Exports_PatSeer_&Gold_Standard/1627__dataset_26_05_26.xlsx


/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  11 patents have no PDF URL (will appear in status report)
  Loaded 1616 patents with valid PDF URLs.
Total patents with valid PDF URL : 1616
Patents with no PDF URL          : 11

Record Types:
Record Type
Application                  1460
Patent                        125
Granted Utility Model          28
Granted Innovation Patent       1
Other                           1
Utility Model Application       1

Legal Status:
Family Legal Status(Dead/Alive)
ALIVE      1257
DEAD        356
UNKNOWN       3

First CPC (codes with ≥ 10 patents):
CPC
B64C29/0033    151
B64D27/34       76
B64C29/00       69
B64C29/0025     58
B64C29/02       45
B64C27/28       32
B64U50/14       26
B64C27/52       25
B64C27/20       25
B64C11/001      24
B64D27/24       23
B64C29/0008     19
B64C27/26       19
B64C27/22       18
Y02T50/60       18
B64D35/08       16
B64U50/19       16
B64C3/56        16
B64C39/08       16
B64C11/28       15
B64C29/0016     13
H02J2105/32     13
B64C27/30       13
B64U2201/20   

In [2]:
# ── CPC definitions ──────────────────────────────────────────────────────
SHROUDED_CPCS   = {"B64C27/20", "B64C11/001", "B64U50/14"}
OPEN_ROTOR_CPCS = {"B64C29/0033", "B64C29/0025"}

first_cpc = (
    df["CPC"].fillna("")
    .str.split(r"\s*;\s*", n=1).str[0]
    .str.strip()
    .str.replace(r"\s+", "", regex=True)
)
df["_first_cpc"] = first_cpc  # kept for review display only

shrouded_df   = df[first_cpc.isin(SHROUDED_CPCS)].copy().reset_index(drop=True)
open_rotor_df = df[first_cpc.isin(OPEN_ROTOR_CPCS)].copy()

print(f"Shrouded candidates  : {len(shrouded_df)}")
print(f"Open rotor candidates: {len(open_rotor_df)}")
assert len(shrouded_df) == 75, f"Expected 75 shrouded, got {len(shrouded_df)}"

# Balanced sample of 75 open rotor patents (seed=42 for reproducibility)
open_rotor_df = open_rotor_df.sample(n=75, random_state=42).reset_index(drop=True)

shrouded_df["category"]   = "shrouded"
open_rotor_df["category"] = "open_rotor"

# Confirm zero overlap
assert not (set(shrouded_df["Record Number"]) & set(open_rotor_df["Record Number"])), \
    "Overlap detected — check CPC definitions"
print("Zero overlap confirmed.")

Shrouded candidates  : 75
Open rotor candidates: 209
Zero overlap confirmed.


In [3]:
from pathlib import Path

OUT_DIR = Path(
    "/home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Drive_files_to_syncronize"
    "/3 - Images Processing & DataSets/1627/Shrouded vs Openrotor"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

selected = pd.concat([shrouded_df, open_rotor_df], ignore_index=True)
csv_out  = OUT_DIR / "selected_patents.csv"
selected[["Record Number", "category", "pdf_url"]].to_csv(csv_out, index=False)
print(f"Saved {len(selected)} patents → {csv_out}")

Saved 150 patents → /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Drive_files_to_syncronize/3 - Images Processing & DataSets/1627/Shrouded vs Openrotor/selected_patents.csv


In [4]:
from IPython.display import display

REVIEW_COLS = ["Record Number", "_first_cpc", "Title", "Assignee"]

print("=== Category counts ===")
print(selected["category"].value_counts().to_string())

print("\n=== Open rotor — all 75 selected (random_state=42) ===")
display(
    open_rotor_df[REVIEW_COLS]
    .rename(columns={"_first_cpc": "first_cpc"})
    .reset_index(drop=True)
)

print("\n=== Shrouded — all 75 ===")
display(
    shrouded_df[REVIEW_COLS]
    .rename(columns={"_first_cpc": "first_cpc"})
    .reset_index(drop=True)
)

=== Category counts ===
category
shrouded      75
open_rotor    75

=== Open rotor — all 75 selected (random_state=42) ===


,Record Number,first_cpc,Title,Assignee
0,US2020255128A1,B64C29/0033,Multicopter with Improved Failsafe Operation,SUPPES FAMILY TRUST (US)
1,US2017121029A1,B64C29/0033,TILT-ROTOR OVER-TORQUE PROTECTION FROM ASYMMET...,BELL HELICOPTER TEXTRON INC (US)
2,US2024025540A1,B64C29/0025,VERTICAL TAKE-OFF AND LANDING AERODYNE OPTIMIS...,TAVIN GERARD (FR)
3,US2021331794A1,B64C29/0033,Aeronautical Apparatus,AERHART LLC (US)
4,US2007158494A1,B64C29/0033,Tilt-rotor aircraft,BURRAGE ROBERT G
...,...,...,...,...
70,EP2570345A1,B64C29/0033,Airplane,EMT INGENIEURGESELLSCHAFT DIPL ING HARTMUT EUE...
71,US2018339772A1,B64C29/0033,Aircraft having Omnidirectional Ground Maneuve...,BELL HELICOPTER TEXTRON INC (US)
72,US2019135423A1,B64C29/0033,BIPLANE TILTROTOR AIRCRAFT,BELL HELICOPTER TEXTRON INC (US)
73,US2019332126A1,B64C29/0033,PROPULSOR TRIM PREDICTION FOR AIRCRAFT,THE BOEING CO (US)



=== Shrouded — all 75 ===


,Record Number,first_cpc,Title,Assignee
0,US2022289371A1,B64C11/001,Line Replaceable Centerbody Assemblies for Duc...,TEXTRON INNOVATIONS INC (US)
1,US2022144422A1,B64C11/001,Modular Device For Propulsion In A Vehicle,WELCEL HUGH BRYAN (US)
2,WO2016028358A2,B64U50/14,High Performance VTOL Aircraft,JUAN CRUZ AYOROA (US)
3,US2021188427A1,B64C27/20,"FLYING OBJECT CONTROL DEVICE, FLYING OBJECT, A...",HONDA MOTOR CO LTD (JP)
4,FR3096959A1,B64U50/14,Protective cover for an aircraft electric moto...,SAFRAN ELECTRICAL & POWER (FR)
...,...,...,...,...
70,US2016159458A1,B64U50/14,PROPELLER,NORTHROP GRUMMAN SYSTEM CORP (US)
71,US2021316849A1,B64U50/14,ROTOR CENTRIFUGAL FORCE RETENTION DEVICE,BELL TEXTRON INC (US)
72,US2022081109A1,B64U50/14,MOTOR-INTEGRATED FLUID MACHINE AND VERTICAL TA...,MITSUBISHI HEAVY IND LTD (JP)
73,US2022063820A1,B64U50/14,MOTOR-INTEGRATED FLUID MACHINE AND VERTICAL TA...,MITSUBISHI HEAVY IND LTD (JP)
